# Отчёт: LLM-оценка блока «Мотивация клиента на оплату»

**Задача:** разработать и проверить prompt-based решение для автоматической оценки диалогов операторов отдела взыскания.

На вход подаются:
- текст диалога оператора и клиента;
- срок просрочки в днях.

На выходе ожидается JSON-оценка: выполнен ли блок **«Мотивация клиента на оплату»**, какая категория клиента определена, какие последствия и нарушения найдены, есть ли исключения.

Этот ноутбук является отчётом по проделанной работе: здесь оставлены основные prompt-версии, функции запуска и тестирования, результаты экспериментов, выявленные проблемы и варианты дальнейшей доработки.

## 1. Постановка задачи

Нужно оценивать, насколько качественно оператор мотивирует клиента на оплату задолженности. Критерий зависит от нескольких признаков:

- срока просрочки и категории клиента;
- согласия или несогласия клиента оплатить;
- последствий, которые озвучил именно оператор;
- допустимости формулировок оператора;
- исключительных ситуаций;
- признака прерванного диалога.

Важный вывод по ходу работы: это не один простой binary classification, а композиция нескольких проверок. Поэтому один монолитный prompt подходит как baseline, но не является оптимальным production-подходом.

## 2. Формализация правил из ТЗ

### Категории клиентов

| Категория | Срок просрочки |
|---|---:|
| A | 1–30 дней |
| Б | 31–45 дней |

### Правила выполнения критерия

| Категория | Статус клиента | Что должен сделать оператор |
|---|---|---|
| A | согласен оплатить | 1 последствие неоплаты, констатация действующего последствия или позитивное уведомление об отмене последствий |
| A | не согласен оплатить | 2 последствия неоплаты или констатация действующих последствий |
| Б | согласен оплатить | 1 последствие именно из категории A |
| Б | не согласен оплатить | 1 последствие из категории A + 1 дополнительное последствие категории Б |

### Последствия категории A

| Последствие |
|---|
| ухудшение кредитной истории |
| штрафы / неустойки |
| звонки |
| выезд сотрудника |
| продажа долга |
| передача в работу коллекторов |

### Дополнительные последствия категории Б

| Последствие |
|---|
| требование полного досрочного погашения |
| ст. 811 ГК РФ |
| продажа долга |
| передача в работу коллекторов |

### Исключения

Критерий считается выполненным автоматически, если упоминается банкротство, ЧС, военные действия, тюрьма, армия, мошенничество, смерть плательщика, инвалидность, лечение, либо если оператор не попрощался в конце диалога.

## 3. Используемая модель и параметры вызова

Использованная модель в рабочем ноутбуке: `gpt-4o-mini` через OpenAI-compatible API endpoint.

Параметры вызова в baseline:

| Параметр | Значение |
|---|---|
| model | `gpt-4o-mini` |
| temperature | `0.0` |
| max_tokens | `700` |
| response_format | `json_object` |
| timeout | `15.0` сек. |
| max_retries клиента | `3` |

Официальные ориентиры для `gpt-4o-mini`: контекстное окно 128K токенов и до 16K выходных токенов на запрос указаны в публикации OpenAI о модели; стоимость в публичном API на момент подготовки отчёта указана как $0.15 за 1M input tokens, $0.075 за cached input и $0.60 за 1M output tokens в официальной документации OpenAI.

Ссылки для проверки актуальных значений:
- https://openai.com/index/gpt-4o-mini-advancing-cost-efficient-intelligence/
- https://developers.openai.com/api/docs/models/gpt-4o-mini

## 4. Выбранный baseline

В качестве baseline был выбран монолитный prompt: все правила, срок просрочки, текст диалога и формат ответа передаются в один LLM-вызов.

Плюсы baseline:

- быстро реализуется;
- легко тестируется;
- не требует отдельной архитектуры;
- подходит для первичной проверки применимости LLM к задаче.

Минусы baseline:

- prompt быстро перегружается;
- правила начинают конкурировать друг с другом;
- модель может путать согласие клиента с качественной мотивацией оператора;
- модель может верно объяснить проблему, но неверно выставить `criterion_met`;
- часть проверок лучше решать детерминированно или отдельными шагами.

## 5. Prompt v1 — минимальный baseline

Ниже приведён компактный baseline prompt как цельный текст, без переменных реализации. В реальном коде вместо `{overdue_days}` и `{dialogue}` подставляются значения конкретного кейса.

```text
Оцени мотивацию оператора на оплату задолженности.

Срок просрочки: {overdue_days} дней

Диалог:
{dialogue}

Категории:
Категория A: 1–30 дней просрочки.
Категория Б: 31–45 дней просрочки.

Последствия:
Последствия категории A:
- ухудшение кредитной истории;
- штрафы / неустойки;
- звонки;
- выезд сотрудника;
- продажа долга;
- передача в работу коллекторов.

Дополнительные последствия категории Б:
- требование полного досрочного погашения;
- ст. 811 ГК РФ;
- продажа долга;
- передача в работу коллекторов.

Исключения:
Критерий считается выполненным автоматически, если:
- в диалоге упоминается банкротство, ЧС, военные действия, тюрьма, армия, мошенничество,
  смерть плательщика, инвалидность или лечение;
- оператор не попрощался в конце диалога.

Как оценивать:
1. Определи категорию клиента по сроку просрочки.
2. Определи, согласен ли клиент оплатить.
3. Проверь, достаточно ли мотивации по категории клиента и статусу клиента.
4. Проверь, нет ли запрещённых формулировок.
5. Последствия и нарушения учитывай только из реплик оператора.
6. Верни итог в JSON.

Формат ответа:
{
  "criterion_met": true,
  "decision": "выполнено",
  "client_category": "A",
  "client_status": "согласен оплатить",
  "exception_detected": false,
  "interrupted_dialog": false,
  "detected_consequences": [],
  "violations": [],
  "explanation": "Краткое объяснение в 1 предложении."
}
```

## 6. Prompt v2 — расширенный baseline с переменными

После первых ошибок prompt был переведён в формат с отдельными переменными: категории, статус клиента, последствия, исключения, правила ролей, правила незачёта и JSON-схема.

Ниже приведена читаемая версия итогового user prompt без Python-переменных.

```text
Оцени, насколько качественно оператор мотивирует клиента на оплату задолженности.

Входные данные:
Срок просрочки: {overdue_days} дней

Диалог:
{dialogue}

Категории клиентов:
- категория A: 1–30 дней просрочки
- категория Б: 31–45 дней просрочки

Как определить согласие клиента:
Согласен:
клиент согласен оплатить задолженность. Согласие может быть выражено явно: «оплачу», «внесу платёж», «закрою сегодня», «да, подтверждаю». Оператор может не называть конкретный срок, если клиент сам заявил намерение оплатить.

Не согласен:
клиент отказывается, торгуется, просит отсрочку, говорит «не могу», «постараюсь», «возможно», или не подтверждает оплату уверенно.

Правила оценки категории A:
- если клиент согласен оплатить: должно быть озвучено 1 последствие неоплаты, констатация действующего последствия или позитивное уведомление об отмене последствий;
- если клиент не согласен оплатить: должно быть озвучено 2 последствия неоплаты или констатация действующих последствий.

Правила оценки категории Б:
- если клиент согласен оплатить: должно быть озвучено 1 последствие именно из категории A;
- если клиент не согласен оплатить: должно быть озвучено минимум 1 последствие из категории A и минимум 1 дополнительное последствие из категории Б.

Последствия категории A:
- ухудшение кредитной истории;
- штрафы / неустойки;
- звонки;
- выезд сотрудника;
- продажа долга;
- передача в работу коллекторов.

Дополнительные последствия категории Б:
- требование полного досрочного погашения;
- ст. 811 ГК РФ;
- продажа долга;
- передача в работу коллекторов.

Формы мотивации:
- последствие неоплаты;
- констатация действующего последствия;
- позитивное уведомление об отмене последствия после оплаты.

Допустимые формулировки оператора:
- вправе;
- может;
- возможно.

Недопустимые формулировки оператора:
- точные даты наступления последствий;
- гарантии будущих последствий, например: «будет», «завтра передадим», «мы передадим»;
- утвердительные формулировки будущих последствий.

Исключения:
- банкротство;
- ЧС;
- военные действия;
- тюрьма;
- армия;
- мошенничество;
- смерть плательщика;
- инвалидность;
- лечение.

Правила автоматического зачёта:
Критерий считается выполненным автоматически, если в диалоге найдено исключение или оператор не попрощался в конце диалога.

Правила ролей:
- последствия и нарушения ищи только в репликах оператора;
- согласие или отказ клиента определяй только по репликам клиента;
- не засчитывай последствия, которые озвучил клиент;
- не придумывай последствия, которых нет в репликах оператора;
- не используй списки правил как найденные последствия в диалоге.

Порядок проверки:
1. Определи категорию клиента только по сроку просрочки.
2. Проверь автоматический зачёт: исключение или отсутствие прощания оператора.
3. Определи статус клиента.
4. Найди последствия, озвученные оператором.
5. Найди нарушения в репликах оператора.
6. Прими решение: сначала автоматический зачёт, затем нарушения, затем достаточность последствий.

Верни только валидный JSON без markdown-разметки и дополнительных комментариев.
```

## 7. Реализация prompt'а в коде

В рабочем варианте prompt хранится не как один длинный текст, а как набор переменных. Это удобнее для экспериментов: можно менять отдельные блоки правил и повторно запускать тесты.

In [ ]:
SYSTEM_PROMPT = """
Ты оцениваешь качество мотивации клиента на оплату задолженности.
Следуй инструкции пользователя.
Верни только валидный JSON.
""".strip()

In [ ]:
overdue_for_A = "категория A: 1–30 дней просрочки"
overdue_for_B = "категория Б: 31–45 дней просрочки"

payment_true = """
клиент согласен оплатить задолженность.
Согласие может быть выражено явно: «оплачу», «внесу платёж», «закрою сегодня», «да, подтверждаю».
Оператор может не называть конкретный срок, если клиент сам заявил намерение оплатить.
""".strip()

payment_false = """
клиент отказывается, торгуется, просит отсрочку, говорит «не могу», «постараюсь», «возможно»,
или не подтверждает оплату уверенно.
""".strip()

eval_rules_A = """
- если клиент согласен оплатить: должно быть озвучено 1 последствие неоплаты,
  констатация действующего последствия или позитивное уведомление об отмене последствий;
- если клиент не согласен оплатить: должно быть озвучено 2 последствия неоплаты
  или констатация действующих последствий.
""".strip()

eval_rules_B = """
- если клиент согласен оплатить: должно быть озвучено 1 последствие именно из категории A;
- если клиент не согласен оплатить: должно быть озвучено минимум 1 последствие из категории A
  и минимум 1 дополнительное последствие из категории Б.

Важно:
- для согласного клиента категории Б недостаточно озвучить только дополнительное последствие категории Б;
- дополнительные последствия категории Б используются как обязательное дополнение только если клиент не согласен оплатить.
""".strip()

motivation_forms = """
Мотивацией считается одно из трёх:

1. Последствие неоплаты:
   например штрафы / неустойки, звонки, ухудшение кредитной истории, выезд сотрудника,
   продажа долга, передача в работу коллекторов, требование полного досрочного погашения,
   ст. 811 ГК РФ.

2. Констатация действующего последствия:
   например уже начисляются штрафы, уже поступают звонки, уже ухудшается кредитная история.

3. Позитивное уведомление об отмене последствия после оплаты:
   например после оплаты звонки прекратятся,
   после оплаты информация по кредитной истории обновится,
   после поступления платежа взаимодействие по задолженности будет прекращено.

Позитивное уведомление об отмене последствия после оплаты засчитывается как мотивация.
""".strip()

consequences_A = """
- ухудшение кредитной истории;
- штрафы / неустойки;
- звонки;
- выезд сотрудника;
- продажа долга;
- передача в работу коллекторов.
""".strip()

consequences_B = """
- требование полного досрочного погашения;
- ст. 811 ГК РФ;
- продажа долга;
- передача в работу коллекторов.
""".strip()

allowed_words = """
- вправе;
- может;
- возможно.
""".strip()

not_allowed_words = """
- точные даты наступления последствий;
- гарантии будущих последствий, например: «будет», «завтра передадим», «мы передадим»;
- утвердительные формулировки будущих последствий.
""".strip()

exceptional_situations = """
- банкротство;
- ЧС;
- военные действия;
- тюрьма;
- армия;
- мошенничество;
- смерть плательщика;
- инвалидность;
- лечение.
""".strip()

auto_pass_rules = """
Критерий считается выполненным автоматически, если:
- в диалоге найдено исключение;
- оператор не попрощался в конце диалога.
""".strip()

role_rules = """
- последствия и нарушения ищи только в репликах оператора;
- согласие или отказ клиента определяй только по репликам клиента;
- не засчитывай последствия, которые озвучил клиент;
- не придумывай последствия, которых нет в репликах оператора;
- не используй списки правил как найденные последствия в диалоге.
""".strip()

fail_rules = """
Критерий считается невыполненным, если:
- оператор не озвучил нужное количество последствий для категории клиента и статуса клиента;
- оператор использовал запрещённую формулировку;
- оператор назвал точную дату наступления последствия;
- оператор гарантировал будущее последствие;
- клиент относится к категории A, а оператор озвучил требование полного досрочного погашения или ст. 811 ГК РФ.
""".strip()

restrictions = """
- оценивай только блок «Мотивация клиента на оплату»;
- не оценивай другие аспекты диалога;
- учитывай только текст диалога и срок просрочки;
- не придумывай факты, последствия, согласие клиента или нарушения;
- ответ «угу» может считаться согласием, если он подтверждает оплату;
- поле detected_consequences заполняй только по явно найденным репликам оператора;
- если последствий нет, detected_consequences должен быть пустым списком [];
- если оператор сказал, что после оплаты последствие прекратится или ситуация улучшится, засчитывай это как мотивацию;
- позитивное уведомление записывай в detected_consequences с type="позитивное уведомление".
""".strip()

decision_steps = """
Порядок проверки:

1. Определи категорию клиента только по сроку просрочки.

2. Проверь автоматический зачёт:
   - исключение;
   - оператор не попрощался в конце диалога.

3. Определи статус клиента:
   - согласен оплатить;
   - не согласен оплатить.

4. Найди последствия, озвученные оператором.

5. Найди нарушения в репликах оператора.

6. Прими решение:
   - сначала проверь автоматический зачёт;
   - затем проверь нарушения;
   - затем проверь, хватает ли последствий по категории клиента и статусу клиента.
""".strip()

output_schema = """
{
  "criterion_met": true,
  "decision": "выполнено",
  "client_category": "A",
  "client_status": "согласен оплатить",
  "exception_detected": false,
  "interrupted_dialog": false,
  "detected_consequences": [],
  "violations": [],
  "explanation": "Краткое объяснение в 1 предложении без прямых цитат из диалога."
}
""".strip()

def build_user_prompt(dialogue: str, overdue_days: int) -> str:
    return f"""
Оцени, насколько качественно оператор мотивирует клиента на оплату задолженности.

Входные данные:
Срок просрочки: {overdue_days} дней

Диалог:
{dialogue}

Категории клиентов:
- {overdue_for_A}
- {overdue_for_B}

Как определить согласие клиента:
Согласен:
{payment_true}

Не согласен:
{payment_false}

Правила оценки категории A:
{eval_rules_A}

Правила оценки категории Б:
{eval_rules_B}

Последствия категории A:
{consequences_A}

Дополнительные последствия категории Б:
{consequences_B}

Формы мотивации:
{motivation_forms}

Допустимые формулировки оператора:
{allowed_words}

Недопустимые формулировки оператора:
{not_allowed_words}

Исключения:
{exceptional_situations}

Правила автоматического зачёта:
{auto_pass_rules}

Правила ролей:
{role_rules}

Правила незачёта:
{fail_rules}

Общие ограничения:
{restrictions}

Порядок проверки:
{decision_steps}

Верни только валидный JSON без markdown-разметки и дополнительных комментариев.

Формат ответа:
{output_schema}
""".strip()

## 8. Функции запуска и тестирования

Ниже оставлены основные функции из рабочего ноутбука:

- `evaluate_dialogue` — вызывает модель и парсит JSON;
- `print_evaluation_result` — красиво выводит один результат;
- `run_evaluation` — прогоняет набор тестовых кейсов и собирает таблицу результатов.

В отчётном ноутбуке эти ячейки оставлены для воспроизводимости, но запуск потребует API-ключ и base URL.

In [ ]:
from openai import OpenAI
import json
import time
import os
import getpass
from typing import Any, Dict
import pandas as pd
from tqdm.auto import tqdm

API_KEY = getpass.getpass("Enter API key: ")
BASE_URL = getpass.getpass("Enter base URL: ")
MODEL_NAME = "gpt-4o-mini"
client = OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=15.0, max_retries=3)

In [ ]:
def evaluate_dialogue(dialogue: str,
                      overdue_days: int,
                      model: str = MODEL_NAME,
                      temperature: float = 0.0,
                      max_tokens: int = 700,
                      max_attempts: int = 3,
                      verbose: bool = True,) -> Dict[str, Any]:

    user_prompt = build_user_prompt(dialogue=dialogue,
                                    overdue_days=overdue_days,)

    start_time = time.perf_counter()
    last_error = None
    raw_content = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                temperature=temperature,
                max_tokens=max_tokens,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": user_prompt},],)

            raw_content = response.choices[0].message.content
            parsed_result = json.loads(raw_content)

            elapsed_time = time.perf_counter() - start_time

            if verbose:
                print("Model:", model)
                print("Base URL:", BASE_URL)
                print("Prompt length:", len(user_prompt))
                print(f"Attempts used: {attempt}")
                print(f"Elapsed time: {elapsed_time:.2f} sec")
                print("Raw answer:\n", raw_content)

            parsed_result["_elapsed_time_sec"] = round(elapsed_time, 2)
            parsed_result["_prompt_length"] = len(user_prompt)
            parsed_result["_model"] = model
            parsed_result["_attempts_used"] = attempt

            return parsed_result

        except Exception as error:
            last_error = error

            if verbose:
                print(f"Attempt {attempt}/{max_attempts} failed: {type(error).__name__}")

    raise RuntimeError(f"Не удалось получить корректный ответ после {max_attempts} попыток.\n"
                       f"Последняя ошибка: {type(last_error).__name__}: {last_error}\n\n"
                       f"Последний raw_content:\n{raw_content}")

In [ ]:
def print_evaluation_result(result: Dict[str, Any]) -> None:
    print("=== Результат оценки ===")
    print(f"Итог: {result.get('decision')}")
    print(f"Критерий выполнен: {result.get('criterion_met')}")
    print(f"Категория клиента: {result.get('client_category')}")
    print(f"Статус клиента: {result.get('client_status')}")
    print(f"Исключение найдено: {result.get('exception_detected')}")
    print(f"Прерванный диалог: {result.get('interrupted_dialog')}")

    print("\n=== Найденные последствия ===")
    consequences = result.get("detected_consequences", [])
    if consequences:
        for idx, consequence in enumerate(consequences, start=1):
            print(f"{idx}. Текст: {consequence.get('text')}")
            print(f"   Тип: {consequence.get('type')}")
            print(f"   Категория: {consequence.get('category')}")
    else:
        print("Не найдены")

    print("\n=== Нарушения ===")
    violations = result.get("violations", [])
    if violations:
        for idx, violation in enumerate(violations, start=1):
            if isinstance(violation, dict):
                print(f"{idx}. Текст: {violation.get('text')}")
                print(f"   Тип: {violation.get('type')}")
                print(f"   Причина: {violation.get('reason')}")
            else:
                print(f"{idx}. {violation}")
    else:
        print("Не найдены")

    print("\n=== Объяснение ===")
    print(result.get("explanation"))

In [ ]:
def run_evaluation(test_cases: list[dict],
                   verbose: bool = False,) -> tuple[pd.DataFrame, float]:

    results = []

    total_start_time = time.perf_counter()

    for case in tqdm(test_cases, desc="Evaluating dialogues"):
        result = evaluate_dialogue(dialogue=case["dialogue"],
                                   overdue_days=case["overdue_days"],
                                   verbose=verbose,)

        results.append({"case_id": case["case_id"],
                        "description": case["description"],
                        "overdue_days": case["overdue_days"],
                        "expected": case["expected"],
                        "predicted": result.get("criterion_met"),
                        "is_correct": result.get("criterion_met") == case["expected"],
                        "decision": result.get("decision"),
                        "client_category": result.get("client_category"),
                        "client_status": result.get("client_status"),
                        "exception_detected": result.get("exception_detected"),
                        "interrupted_dialog": result.get("interrupted_dialog"),
                        "detected_consequences": result.get("detected_consequences"),
                        "violations": result.get("violations"),
                        "elapsed_time_sec": result.get("_elapsed_time_sec"),
                        "explanation": result.get("explanation"),})

    total_elapsed_time = time.perf_counter() - total_start_time

    results_df = pd.DataFrame(results)

    return results_df, total_elapsed_time

## 9. Тестовые данные

В тестировании использовались:

1. примеры из ТЗ;
2. несколько ручных длинных кейсов;
3. реалистичный контрольный тест из 10 диалогов;
4. диагностические наборы по отдельным параметрам.

Ниже оставлены основные тестовые наборы из рабочего ноутбука.

In [ ]:
test_cases_tz = [
    {
        "case_id": "tz_01_A_agree_positive_notification",
        "description": "Пример 1 из ТЗ: категория A, клиент согласен, позитивное уведомление",
        "overdue_days": 10,
        "expected": True,
        "dialogue": """
Клиент: Сегодня оплачу.
Оператор: Как только платеж поступит, звонки прекратятся.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_02_A_agree_no_motivation",
        "description": "Пример 2 из ТЗ: категория A, клиент согласен, нет мотивации",
        "overdue_days": 10,
        "expected": False,
        "dialogue": """
Клиент: Сегодня оплачу.
Оператор: Ждём платежа, себя не подводите.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_03_A_wrong_period",
        "description": "Пример 3 из ТЗ: категория A, ошибка периода",
        "overdue_days": 10,
        "expected": False,
        "dialogue": """
Оператор: Банк может потребовать полное досрочное погашение и передать долг коллекторам.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_04_A_disagree_two_consequences",
        "description": "Пример 4 из ТЗ: категория A, клиент не согласен, 2 последствия",
        "overdue_days": 22,
        "expected": True,
        "dialogue": """
Клиент: Не могу сегодня.
Оператор: Банк вправе ежедневно начислять штрафы — это переплата, долг растёт.
Оператор: Также могут быть звонки вам и контактным лицам, создавая неудобства.
Оператор: Как планируете решать?
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_05_A_forbidden_assertive_wording",
        "description": "Пример 5 из ТЗ: точная дата последствия и гарантия",
        "overdue_days": 15,
        "expected": False,
        "dialogue": """
Оператор: Если не оплатите до 13 апреля, мы передадим дело коллекторам.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_06_exception_treatment",
        "description": "Пример 6 из ТЗ: исключение — лечение",
        "overdue_days": 18,
        "expected": True,
        "dialogue": """
Клиент: У меня инсульт, я на лечении.
Оператор: Понимаю, выздоравливайте.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_08_implicit_agreement_ugu",
        "description": "Пример 8 из ТЗ: «угу» как согласие",
        "overdue_days": 10,
        "expected": True,
        "dialogue": """
Оператор: Подтверждаете оплату до пятницы?
Клиент: Угу.
Оператор: Отлично, после оплаты обновим кредитную историю.
Оператор: До свидания.
""".strip(),
    },
]

manual_case_tz_words = {
    "case_id": "manual_01_tz_words_complex",
    "description": "Ручной длинный кейс: формулировки близки к ТЗ, проверка нескольких признаков",
    "overdue_days": 22,
    "expected": True,
    "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность, нужно обсудить оплату.
Клиент: Сегодня не могу оплатить, денег нет.
Оператор: Правильно понимаю, что сегодня оплатить не получится?
Клиент: Да, сегодня не смогу.
Оператор: Банк вправе ежедневно начислять штрафы — это переплата, долг растёт.
Оператор: Также могут быть звонки вам и контактным лицам, создавая неудобства.
Оператор: Как планируете решать вопрос с оплатой?
Клиент: Постараюсь внести платёж завтра или послезавтра.
Оператор: Хорошо, ожидаем оплату. До свидания.
""".strip(),
}

manual_case_synonyms = {
    "case_id": "manual_02_synonyms_complex",
    "description": "Ручной длинный кейс: естественные синонимы вместо точных формулировок",
    "overdue_days": 36,
    "expected": True,
    "dialogue": """
Оператор: Добрый день. Звоню по вопросу просроченной задолженности. Когда сможете внести платёж?
Клиент: Сейчас не готов платить, нужна отсрочка хотя бы на пару недель.
Оператор: То есть в предложенный срок оплату подтвердить не можете?
Клиент: Да, пока не могу.
Оператор: В таком случае банк может продолжить начисление неустойки, и сумма задолженности станет больше.
Оператор: Также банк вправе потребовать вернуть всю сумму задолженности раньше установленного графика.
Клиент: Понял, но сейчас всё равно денег нет.
Оператор: Какой срок оплаты вы можете назвать сейчас?
Клиент: Попробую решить вопрос в течение недели.
Оператор: Спасибо за информацию. До свидания.
""".strip(),
}

test_cases = test_cases_tz + [manual_case_tz_words,
                              manual_case_synonyms,]

In [ ]:
realistic_test_cases = [
    {
        "case_id": "real_01_A_agree_positive_notification",
        "description": "Категория A, клиент согласен, оператор даёт позитивное уведомление",
        "overdue_days": 7,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Звоню по вопросу просроченного платежа по договору.
Клиент: Да, я видел уведомление, сегодня внесу оплату.
Оператор: Правильно понимаю, что платёж будет сегодня?
Клиент: Да, сегодня вечером.
Оператор: После поступления платежа звонки по задолженности прекратятся.
Клиент: Хорошо, понял.
Оператор: Спасибо за информацию. До свидания.
""".strip(),
    },
    {
        "case_id": "real_02_A_agree_no_motivation",
        "description": "Категория A, клиент согласен, но оператор не мотивирует",
        "overdue_days": 11,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. Напоминаю о просроченной задолженности.
Клиент: Да, я в курсе, сегодня постараюсь оплатить.
Оператор: Хорошо, тогда ожидаем платёж. Пожалуйста, не затягивайте.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_03_A_disagree_two_A_consequences",
        "description": "Категория A, клиент не согласен, оператор называет два последствия A",
        "overdue_days": 23,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. У вас сохраняется просроченная задолженность. Когда сможете оплатить?
Клиент: Сейчас не могу, денег нет, возможно только через неделю.
Оператор: Понимаю. При отсутствии оплаты банк вправе начислять неустойку, из-за чего сумма долга может увеличиваться.
Оператор: Также могут продолжаться звонки вам и контактным лицам по вопросу задолженности.
Клиент: Понял, но раньше оплатить не получится.
Оператор: Какой срок оплаты можете назвать сейчас?
Клиент: Попробую решить вопрос до конца недели.
Оператор: Спасибо, информацию зафиксировал. До свидания.
""".strip(),
    },
    {
        "case_id": "real_04_A_wrong_period_full_repayment",
        "description": "Категория A, оператор использует последствие категории Б",
        "overdue_days": 19,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. По вашему договору есть просроченный платёж.
Клиент: Я понимаю, но сейчас оплатить не могу.
Оператор: В случае дальнейшей неоплаты банк может потребовать полного досрочного погашения всей задолженности.
Клиент: Но просрочка ведь меньше месяца.
Оператор: Я обязан предупредить о возможных последствиях.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_05_B_agree_A_consequence",
        "description": "Категория Б, клиент согласен, оператор называет последствие категории A",
        "overdue_days": 34,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность более месяца.
Клиент: Да, понимаю, завтра смогу внести платёж.
Оператор: Подтверждаете оплату завтра?
Клиент: Да, завтра оплачу.
Оператор: До поступления оплаты могут продолжаться звонки по задолженности.
Клиент: Хорошо, понял.
Оператор: Спасибо. До свидания.
""".strip(),
    },
    {
        "case_id": "real_06_B_agree_only_B_consequence",
        "description": "Категория Б, клиент согласен, но оператор называет только дополнительное последствие Б",
        "overdue_days": 39,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. У вас длительная просрочка по договору.
Клиент: Я смогу оплатить до пятницы.
Оператор: То есть оплату до пятницы подтверждаете?
Клиент: Да, подтверждаю.
Оператор: Банк может потребовать полного досрочного погашения задолженности.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_07_B_disagree_A_and_B_consequences",
        "description": "Категория Б, клиент не согласен, оператор называет последствие A и Б",
        "overdue_days": 41,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить просроченную задолженность?
Клиент: Пока не могу сказать, денег нет.
Оператор: В таком случае банк вправе начислять неустойку, и сумма задолженности может увеличиваться.
Оператор: Также возможно применение ст. 811 ГК РФ.
Клиент: Я понял, но сейчас всё равно оплатить не смогу.
Оператор: Как планируете решать вопрос?
Клиент: Буду искать деньги, но срок назвать не могу.
Оператор: Информацию зафиксировал. До свидания.
""".strip(),
    },
    {
        "case_id": "real_08_forbidden_exact_date_and_guarantee",
        "description": "Запрещённая формулировка: точная дата и гарантия последствия",
        "overdue_days": 16,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Я смогу оплатить только позже, сейчас денег нет.
Оператор: Если не оплатите до 20 мая, 21 мая мы передадим задолженность коллекторам.
Клиент: Я понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_09_exception_bankruptcy",
        "description": "Исключение: банкротство",
        "overdue_days": 27,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете внести оплату по задолженности?
Клиент: Сейчас не могу обсуждать оплату, я прохожу процедуру банкротства.
Оператор: Документы по процедуре уже поданы?
Клиент: Да, этим занимается юрист.
Оператор: Понял, информацию зафиксирую. До свидания.
""".strip(),
    },
    {
        "case_id": "real_10_interrupted_dialog_no_goodbye",
        "description": "Прерванный диалог: оператор не попрощался",
        "overdue_days": 21,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Пока не могу, возможно на следующей неделе.
Оператор: При отсутствии оплаты банк вправе начислять неустойку, сумма задолженности может увеличиваться.
Оператор: Какой срок оплаты можете назвать?
""".strip(),
    },
]

## 10. Результаты основных экспериментов

Сводка по основным прогонам:

| Эксперимент | Кейсов | Корректных | Accuracy | Общее время | Среднее время | Макс. время |
|---|---|---|---|---|---|---|
| Проверка на примерах из ТЗ + 2 ручных кейса | 9 | 6 | 66.67% | 174.07 сек | 19.34 сек | 61.10 сек |
| Промежуточная модификация prompt'а | 9 | 5 | 55.56% | 198.73 сек | 22.08 сек | 45.07 сек |
| Одна из лучших промежуточных версий | 9 | 7 | 77.78% | 167.41 сек | 18.60 сек | 60.07 сек |
| Реалистичный контрольный тест | 10 | 6 | 60.00% | 230.54 сек | 23.05 сек | — |

### Ошибки финального baseline на примерах из ТЗ

| case_id | Описание | Expected | Predicted | Комментарий |
|---|---|---|---|---|
| tz_02_A_agree_no_motivation | Пример 2 из ТЗ: клиент согласен, нет мотивации | False | True | Модель считает блок выполненным из-за согласия клиента, хотя оператор не озвучил мотивацию. |
| tz_03_A_wrong_period | Пример 3 из ТЗ: ошибка периода | False | True | Модель не блокирует критерий при использовании последствия категории Б для категории A. |
| tz_05_A_forbidden_assertive_wording | Пример 5 из ТЗ: точная дата и гарантия | False | True | Модель видит последствие, но не фиксирует запрещённую формулировку. |

### Ошибки реалистичного контрольного теста

| case_id | Описание | Expected | Predicted | Комментарий |
|---|---|---|---|---|
| real_02_A_agree_no_motivation | Категория A, клиент согласен, но оператор не мотивирует | False | True | Согласие клиента ошибочно трактуется как выполненная мотивация. |
| real_04_A_wrong_period_full_repayment | Категория A, оператор использует последствие категории Б | False | True | Ошибка периода не заблокировала итоговый критерий. |
| real_06_B_agree_only_B_consequence | Категория Б, клиент согласен, только последствие Б | False | True | При согласии клиента категории Б требуется последствие из A, а не только дополнительное Б. |
| real_08_forbidden_exact_date_and_guarantee | Точная дата и гарантия последствия | False | True | Запрещённая формулировка не попала в violations. |

## 11. Диагностические тесты по отдельным параметрам

После нестабильных результатов на общем тесте задача была разложена на отдельные признаки. Это помогло понять, где именно baseline теряет качество.

| Параметр | Кейсов | Accuracy | Основной вывод |
|---|---|---|---|
| Наличие мотивации | 10 | 50.00% | модель путает ожидание платежа с мотивацией; иногда не засчитывает позитивное уведомление |
| Статус клиента | 10 | 80.00% | не всегда уверенно различает намерение, обещание и просьбу об отсрочке |
| Категория клиента | 10 | 100.00% | категория по сроку просрочки определяется стабильно |
| Соответствие мотивации категории и статусу клиента | 10 | 50.00% | ошибки в правилах A/Б и в достаточности последствий |
| Детекция исключений | 10 | 20.00% | не все исключения из ТЗ стабильно распознаются как автозачёт |
| Прерванный диалог | 10 | 50.00% | не всегда корректно обрабатывается отсутствие прощания |
| Детекция нарушений | 10 | 40.00% | слабо ловятся гарантии будущих последствий и точные даты |
| Извлечение озвученных последствий | 10 | 50.00% | иногда придумываются последствия или игнорируется явная мотивация |
| Тип нарушения | 10 | 70.00% | тип нарушения часто угадывается лучше, чем сам факт нарушения |
| Тип исключения | 10 | 10.00% | тип исключения почти не извлекается в текущем JSON-формате |
| Понимание ролей Оператор / Клиент | 10 | 10.00% | модель часто смешивает последствия из реплик клиента и оператора |

## 12. Основные выявленные проблемы

### 1. Согласие клиента ≠ мотивация оператора

Модель часто считает критерий выполненным, если клиент согласился оплатить. Но по ТЗ оценивается не сам факт согласия клиента, а то, **мотивировал ли оператор**: озвучил ли последствие, констатацию действующего последствия или позитивное уведомление.

Пример системной ошибки: клиент говорит, что оплатит, оператор отвечает «ждём платёж», а модель ставит `criterion_met=true`.

### 2. Ошибка периода не всегда блокирует критерий

Для категории A нельзя использовать требование полного досрочного погашения или ст. 811 ГК РФ. Baseline иногда объясняет это как проблему, но всё равно возвращает `criterion_met=true`.

### 3. Запрещённые формулировки недостаточно стабильны

Модель плохо фиксирует точные даты наступления последствий и гарантии будущего последствия, особенно если одновременно видит допустимое последствие из списка.

### 4. Поля ответа могут конфликтовать

В некоторых случаях `decision` и `criterion_met` расходились по смыслу: модель могла написать `decision="не выполнено"`, но вернуть `criterion_met=true`.

### 5. Монолитный prompt перегружается

Попытки добавить больше уточнений, примеров и критических правил не всегда улучшали качество. В части прогонов качество падало с 77.78% до 55.56% или 66.67%.

### 6. Производительность и стоимость

Даже для небольших тестовых наборов один большой prompt давал среднее время ответа около 15–30 секунд на кейс. Это делает монолитный подход неудобным для production без оптимизации контекста и архитектуры.

## 13. Почему один prompt недостаточен

Задача содержит несколько зависимых проверок:

1. категория клиента по сроку просрочки;
2. статус клиента;
3. извлечение последствий из реплик оператора;
4. проверка допустимости формулировок;
5. исключения и прерванный диалог;
6. финальное решение.

Монолитный prompt заставляет модель держать в контексте все правила сразу. В итоге модель иногда применяет не тот блок правил, смешивает роли «оператор» и «клиент», или ориентируется на общий исход диалога вместо формального критерия.

Главный вывод: **монолитный prompt подходит как baseline, но не является оптимальным production-решением**.

## 14. Варианты дальнейшей доработки

### 14.1. Гибридный pipeline: правила + LLM

Часть проверок можно вынести из LLM в детерминированный код:

- определение категории по сроку просрочки;
- проверка отсутствия прощания;
- поиск точных дат;
- проверка отдельных запрещённых слов;
- базовая агрегация итогового решения.

LLM при этом использовать для более семантических задач:

- извлечение последствий из естественного текста;
- определение согласия/отказа клиента;
- распознавание синонимов последствий;
- выделение исключительных ситуаций.

### 14.2. Разделение задачи на несколько LLM-вызовов

Вместо одного prompt'а можно сделать последовательность простых запросов:

```text
1. Определи категорию клиента.
2. Определи статус клиента.
3. Извлеки последствия только из реплик оператора.
4. Найди нарушения в репликах оператора.
5. Найди исключения.
6. Собери итоговое решение по правилам.
```

Такой подход уменьшит контекст каждого запроса и упростит тестирование каждого этапа.

### 14.3. Structured reasoning / CoT-подход

Можно использовать структурированный промежуточный JSON:

```json
{
  "category_analysis": "...",
  "client_status_analysis": "...",
  "motivation_analysis": "...",
  "violations_analysis": "...",
  "final_decision": true
}
```

Важно: внутренние рассуждения не обязательно показывать пользователю, но структурные поля помогают модели не пропускать этапы проверки.

### 14.4. ReAct / rule lookup

Модель может сначала определить категорию и статус клиента, а затем обращаться только к нужному набору правил:

```text
A + согласен → правило A_agree
A + не согласен → правило A_disagree
Б + согласен → правило B_agree
Б + не согласен → правило B_disagree
```

Это снижает шум и риск применения не того блока правил.

### 14.5. Отдельная функция для категории

Категория клиента полностью определяется сроком просрочки, поэтому её лучше считать кодом:

```python
if 1 <= overdue_days <= 30:
    category = "A"
elif 31 <= overdue_days <= 45:
    category = "Б"
else:
    category = "unknown"
```

Это уменьшит нагрузку на LLM и уберёт один источник ошибки.

### 14.6. Проверка других моделей

Дополнительно можно протестировать:

- GigaChat;
- более сильную OpenAI-модель;
- локальную/корпоративную модель;
- разные языки инструкций: русский vs английский system prompt.

### 14.7. Демо-интерфейс

Для демонстрации решения можно сделать:

- Streamlit-приложение;
- Flask API;
- форму загрузки диалога и срока просрочки;
- вывод JSON и человекочитаемого отчёта;
- подсветку найденных последствий и нарушений.

## 15. Ограничения текущего решения

- Диалоги считаются уже транскрибированными.
- Роли `Оператор` и `Клиент` считаются размеченными в тексте.
- Не учитываются аудио-признаки: интонация, перебивания, паузы, эмоции.
- Решение тестировалось на небольшом наборе кейсов; нужна расширенная размеченная выборка.
- Prompt и тесты ориентированы на русский язык.
- Один большой prompt увеличивает стоимость и время ответа.
- LLM может возвращать верное объяснение, но ошибочный boolean-результат.
- Для production нужна валидация на реальных диалогах и мониторинг качества.

## 16. Итоговый вывод

Baseline на одном prompt'е позволяет быстро получить рабочий прототип и проверить применимость LLM к задаче оценки мотивации в диалогах взыскания.

Однако результаты тестирования показывают, что задача слишком составная для надёжного решения одним монолитным prompt'ом. Модель часто путает согласие клиента с качеством мотивации оператора, не всегда фиксирует запрещённые формулировки и может нестабильно извлекать последствия.

Наиболее перспективное направление развития — гибридная архитектура:

```text
детерминированные правила + LLM extraction + финальная агрегация решения
```

Такой подход должен снизить размер prompt'а, повысить стабильность, ускорить обработку и упростить диагностику ошибок.

## Appendix A. Результаты из рабочего ноутбука

Ниже приведены текстовые итоги ключевых прогонов, сохранённые из рабочего ноутбука.

```text
=== Итоги тестирования ===
Всего кейсов: 9
Корректных ответов: 6
Accuracy: 66.67%
Общее время: 174.07 сек.
Среднее время на кейс: 19.34 сек.
Минимальное время: 11.66 сек.
Максимальное время: 61.10 сек.
```

```text
=== Реалистичный контрольный тест ===
Всего кейсов: 10
Корректных ответов: 6
Accuracy: 60.00%
Общее время: 230.54 сек.
Среднее время на кейс: 23.05 сек.
```

```text
=== Проверка определения категории клиента ===
Всего кейсов: 10
Корректных категорий: 10
Accuracy по категории: 100.00%
```

```text
=== Проверка статуса клиента ===
Всего кейсов: 10
Корректных ответов: 8
Accuracy по статусу клиента: 80.00%
```

```text
=== Проверка понимания ролей Оператор / Клиент ===
Всего кейсов: 10
Accuracy по статусу клиента: 90.00%
Accuracy по последствиям из реплик оператора: 10.00%
Accuracy по нарушениям из реплик оператора: 90.00%
Общая accuracy по пониманию ролей: 10.00%
```